# FedCardioTwin — Kaggle T4 Notebook

**Continually Updated, Uncertainty-Aware Federated Digital Twins for
Patient-Specific Cardiac Monitoring Across Heterogeneous Hospitals**
(IEEE J-BHI special issue)

**Dataset required:** attach `hammad0110/ieee-2` as input before running.

**Session strategy (Kaggle 12 h limit):**
- Each FULL federated cell runs 1–2 strategies × 3 seeds (~2–4 h each).
- Hit **Save & Run All** at a checkpoint to commit `/kaggle/working/` as output.
- Next session: attach that committed output as a second input to resume.

In [ ]:
# 1. Clone repo + install deps
import os, sys, subprocess

REPO_URL = 'https://github.com/perfectenshclag/FedCardioTwin-.git'

if os.path.exists('fedcardiotwin'):
    REPO_ROOT = os.path.abspath('.')
elif os.path.exists('/kaggle/working/FedCardioTwin/fedcardiotwin'):
    REPO_ROOT = '/kaggle/working/FedCardioTwin'
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                    '/kaggle/working/FedCardioTwin'], check=True)
    REPO_ROOT = '/kaggle/working/FedCardioTwin'

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
!pip install -q -r requirements.txt

if not os.path.exists('external/evaluation-2021'):
    !git clone -q --depth 1 https://github.com/physionetchallenges/evaluation-2021 external/evaluation-2021

import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU)')

In [ ]:
# 2. Kaggle path setup
# Links the read-only dataset cache and writable output dirs into the repo tree.
import os, shutil

CACHE_SRC  = '/kaggle/input/ieee-2/cache'
CKPT_DIR   = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'

for d in [CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Symlink cache (read-only is fine for training)
if not os.path.exists('data'):
    os.makedirs('data')
if not os.path.islink('data/cache') and not os.path.exists('data/cache'):
    os.symlink(CACHE_SRC, 'data/cache')
    print('Linked cache ->', CACHE_SRC)

# Symlink checkpoints + results so relative paths in scripts work
for name, target in [('checkpoints', CKPT_DIR), ('results', RESULTS_DIR)]:
    if not os.path.islink(name) and not os.path.exists(name):
        os.symlink(target, name)

# --- Resume from a previously committed output ---
# If you re-attach a previous run's output as input (e.g. ieee-2-run),
# this copies the saved checkpoints so training picks up where it left off.
PREV = '/kaggle/input/ieee-2-run/checkpoints'
if os.path.isdir(PREV):
    copied = 0
    for f in os.listdir(PREV):
        dst = os.path.join(CKPT_DIR, f)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(PREV, f), dst)
            copied += 1
    if copied:
        print(f'Resumed {copied} checkpoint file(s) from previous run.')

print('Cache :', CACHE_SRC)
print('Checkpoints :', CKPT_DIR)
print('Results :', RESULTS_DIR)
print('CWD :', os.getcwd())

In [ ]:
# 3. Smoke test — full pipeline on synthetic data (~2 min)
!python tests/smoke_test.py

In [ ]:
# 4. Cache sanity check
import numpy as np

CLIENTS = ['CPSC', 'CPSC-Extra', 'Georgia', 'Chapman', 'Ningbo', 'PTB-XL', 'PTBXL_TRACKB']
for c in CLIENTS:
    X = np.load(f'data/cache/{c}/X.npy', mmap_mode='r')
    Y = np.load(f'data/cache/{c}/Y.npy', mmap_mode='r')
    print(f'{c:16s} X={str(X.shape):20s} {X.dtype}   Y={str(Y.shape):18s} {Y.dtype}')
!du -sh data/cache/*

In [ ]:
# 5. FAST sanity run (~15 min) — confirms learning before the overnight runs
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage centralized --preset fast {C}
!python scripts/run_experiments.py --stage federated --preset fast --strategies fedavg fedbn {C}

In [ ]:
# 6a. Pooled upper bound — centralized, 3 seeds (~1–2 h)
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage centralized --preset full {C}

In [ ]:
# 6b-i. Federated: local + FedAvg (~2–4 h)
# Run one session at a time; completed seeds auto-skip on re-run.
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage federated --preset full --strategies local fedavg {C}

In [ ]:
# 6b-ii. Federated: FedProx + FedBN + FedAvgM (~3–5 h)
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage federated --preset full --strategies fedprox fedbn fedavgm {C}

In [ ]:
# 6b-iii. Federated: Ditto + FedPer (~2–3 h)
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage federated --preset full --strategies ditto fedper {C}

In [ ]:
# 6c. Twin loop + Conformal + Leave-one-hospital-out (~3–5 h total)
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!python scripts/run_experiments.py --stage twin     --preset full {C}
!python scripts/run_experiments.py --stage conformal              {C}
!python scripts/run_experiments.py --stage loho     --preset full {C}

In [ ]:
# 6d. Seed ensemble — headline number
!python scripts/ensemble_eval.py \
    --ckpts {CKPT_DIR}/central_seed0.pt \
            {CKPT_DIR}/central_seed1.pt \
            {CKPT_DIR}/central_seed2.pt \
    --results-dir {RESULTS_DIR}

In [ ]:
# 6e. Paper tables (mean ± std + Wilcoxon vs FedAvg)
!python scripts/make_tables.py

## Output → paper mapping

| Result file | Paper artifact |
|---|---|
| `results/fed_*_seed*.json` | Table 2: per-client + MEAN/WORST AUROC/F1, comm MB/round; `history` → Fig AUROC vs rounds |
| `results/centralized_seed*.json` | Table 2 top row: pooled upper bound |
| `results/ensemble.json` | Table 2 headline: 3-seed SE-Inception ensemble |
| `results/loho_seed*.json` | Leave-one-hospital-out generalization table |
| `results/twin_seed*.json` | Table 3: cold vs warm AUROC, gain curve, alert AUROC, backward transfer |
| `results/twin_seed*_arrays.npz` | Raw per-record arrays for post-hoc conformal coverage analysis |
| `results/conformal.json` | Table 4: per-hospital FNR coverage + set size (local vs federated λ) |
| `make_tables.py` output | mean ± std + Wilcoxon signed-rank significance vs FedAvg |

**Ablations (Table 5):** toggle `se=False`, `augment=False`, `mixup_alpha=0`, `ema_decay=0` in
`configs.py`, or pass `--strategies fedavg` vs `fedbn`, or set `update_steps=0`/`replay_size=1` for twin.